# Week 2 skills notebook: reading code you did not write

This notebook belongs to the skills track. It uses a made-up fruit cooperative, so the skills stay separate from this week's project.

An AI assistant writes code faster than anyone can read it, so reading becomes the skill that counts. This week you predict what code will do before you run it, write the unit of every number, find where a program starts, follow one value from the first step to the last, and catch code whose units do not fit.

**How to use it:** run each cell in order. When a section says *Predict first*, write down what you expect before you run the cell.

## 1. A name holds a value

A line such as `crates = 40` stores the value 40 under the name `crates`. Python works out the right side first and stores the result under the name on the left. From then on, the name stands for the value.

Each line below ends with a comment after `#` that gives the unit of the name. Python skips comments. They are there for the reader.

**Predict first:** what will the last line print?

In [ ]:
crates = 40              # crates
kg_per_crate = 18        # kilograms per crate
price_per_kg = 0.9       # dollars per kilogram

kg = crates * kg_per_crate          # crates x kg per crate = kg
revenue = kg * price_per_kg         # kg x dollars per kg = dollars
print(f"{crates} crates weigh {kg} kg and sell for {revenue} dollars")

The `f` before the quotes makes an f-string. Python replaces each name inside curly braces with its value, which is how a program reports a result in a sentence.

**Change one line.** Change `crates = 40` to `crates = 55` and run the cell again. Every line that uses `crates` changes with it. A good program therefore states each input once, near the top.

## 2. A list, a loop and a running total

A list holds values in order. Positions count from zero, so `weights[0]` is the first value.

A `for` loop runs its indented lines once for each value in the list. The indent marks what repeats. The line `total = total + w` is the running-total pattern: `total` starts at zero and grows by one value on each turn. The `print` inside the loop shows `total` after every turn, which is the simplest way to watch a program work.

**Predict first:** what is the total after the third turn?

In [ ]:
weights = [18, 20, 17, 19]      # kilograms in four crates
print("first crate:", weights[0], "kg")

total = 0
for w in weights:
    total = total + w
    print(f"after adding {w} kg, the total is {total} kg")
print("total:", total, "kg")

## 3. Your turn: predict, then check

Read this program without running it. It keeps two running totals, one for days with sales and one for units sold, and it prints the average over the days with sales.

```python
counts = [3, 0, 5, 2, 0, 4]
days_with_sales = 0
units = 0
for c in counts:
    if c > 0:
        days_with_sales = days_with_sales + 1
        units = units + c
print(units / days_with_sales)
```

The line `if c > 0:` lets the two lines indented under it run only when `c` is above zero. Work out by hand what the program prints: write a small table with one row per turn and the values of `days_with_sales` and `units` after each turn. Put your answer in `my_prediction` in the next cell and run it. The CHECK cell after it runs the program and compares.

In [ ]:
# your turn 1: replace None with the number you predict
my_prediction = None

In [ ]:
# CHECK
counts = [3, 0, 5, 2, 0, 4]
days_with_sales = 0
units = 0
for c in counts:
    if c > 0:
        days_with_sales = days_with_sales + 1
        units = units + c
result = units / days_with_sales
assert my_prediction is not None, "my_prediction is still empty"
assert abs(my_prediction - result) < 1e-9, f"not yet: the program prints {result}. Trace it again, one turn at a time."
print("CHECK passed: the program prints", result)

## 4. Read a program for its shape

Below is a longer program an AI assistant wrote for a fruit cooperative. Four farms deliver crates every week, and the program reports how many kilograms each farm delivered.

Do not start at the first line and read every word. Read for shape first, and answer three questions on paper:

1. **What goes in?** Look for where the data comes from.
2. **What comes out?** Look for what the program prints or returns.
3. **What are the main steps?** Each `def` block defines one step, called a function, and its name and the sentence under it say what it does.

A function is a named block of steps. The names in brackets after `def` are its inputs, and the line that starts with `return` hands back its output.

Then decide what you can leave unread. For the question *how many kilograms did each farm deliver?*, the inside of `make_deliveries` does not matter, because it only makes the practice data. Three lines matter: the `kg` line in `add_weight`, the `groupby` line in `farm_totals`, and the line that calls them. Run the cell.

In [ ]:
# Written by an AI assistant: kilograms delivered by each farm
import numpy as np
import pandas as pd

def make_deliveries(seed):
    """Make 12 weeks of made-up deliveries from four farms."""
    rng = np.random.default_rng(seed)
    crops = {"A": "oranges", "B": "oranges", "C": "pineapples", "D": "limes"}
    kg_per_crate = {"oranges": 18, "pineapples": 12, "limes": 9}
    price_per_kg = {"oranges": 0.9, "pineapples": 1.4, "limes": 2.2}
    rows = []
    for week in range(1, 13):
        for farm, crop in crops.items():
            rows.append({"week": week, "farm": farm, "crop": crop,
                         "crates": int(rng.integers(10, 40)),
                         "kg_per_crate": kg_per_crate[crop],
                         "price_per_kg": price_per_kg[crop]})
    return pd.DataFrame(rows)

def add_weight(deliveries):
    """Add a column kg: crates times kilograms per crate."""
    deliveries = deliveries.copy()
    deliveries["kg"] = deliveries["crates"] * deliveries["kg_per_crate"]
    return deliveries

def farm_totals(deliveries):
    """Total kilograms per farm, largest first."""
    return deliveries.groupby("farm")["kg"].sum().sort_values(ascending=False)

## 5. Find where it starts

The cell above printed nothing, and nothing went wrong. A `def` block teaches Python a new function and stops there. The steps inside it run only when a line calls the function by name.

So where does the work begin? Look for the first line that sits outside every `def` block and is not an import. That line is the entry point. In the cell below, the entry point is the first line: it calls `make_deliveries`, passes the result to `add_weight`, and stores what comes back under the name `deliveries`.

Read a nested call such as `add_weight(make_deliveries(seed=11))` from the inside out. The inner function runs first, and its output becomes the outer function's input.

Read `deliveries.groupby("farm")["kg"].sum()` from left to right: take the table, split it into groups by farm, take the `kg` column of each group, and add it up.

In [ ]:
deliveries = add_weight(make_deliveries(seed=11))
print("rows and columns:", deliveries.shape)
print(deliveries.head(8))
print()
totals = farm_totals(deliveries)
print("kilograms delivered per farm:")
print(totals)

## 6. Follow one value through the program

Reading for shape tells you what a program is for. Following one value tells you what it does. Take farm C in week 3, and print it after each step.

The line `raw[(raw["week"] == 3) & (raw["farm"] == "C")]` keeps the rows where both conditions hold. `==` asks whether two values are equal, and `&` means *and*.

Experienced programmers trace with `print` every day, because a printed value replaces a belief about the code with an observation of it.

In [ ]:
raw = make_deliveries(seed=11)
step1 = raw[(raw["week"] == 3) & (raw["farm"] == "C")]
print("step 1, the raw delivery:")
print(step1)

step2 = add_weight(raw)
row = step2[(step2["week"] == 3) & (step2["farm"] == "C")]
print()
print("step 2, after add_weight:", int(row["crates"].iloc[0]), "crates x", int(row["kg_per_crate"].iloc[0]), "kg =", int(row["kg"].iloc[0]), "kg")

farm_c = step2[step2["farm"] == "C"]
print()
print("step 3, farm C's twelve weeks add up to", int(farm_c["kg"].sum()), "kg")
print("farm_totals says farm C delivered", int(farm_totals(step2)["C"]), "kg")

## 7. Your turn: trace farm B

Write code that adds up farm B's kilograms yourself, without calling `farm_totals`, and store the result in `farm_b_kg`. Ask your AI assistant for help if you need it, and read its code before you run it. Then run the CHECK cell, which compares your number with the program's.

In [ ]:
# your turn 2: farm B's kilograms, added up without farm_totals
farm_b_kg = None

In [ ]:
# CHECK
assert farm_b_kg is not None, "farm_b_kg is still empty"
assert farm_b_kg == totals["B"], "your total differs from farm_totals: print the rows you added up"
print("CHECK passed: farm B delivered", int(farm_b_kg), "kg")

## 8. Read the AI's code before you run it

We asked the AI assistant: *"Add a function that computes each farm's revenue in dollars."* It wrote the function below.

Do not run it yet. Write the unit of each column the formula uses: `crates` counts crates, and `price_per_kg` is in dollars per kilogram. Then multiply the units the way the code multiplies the columns. Do you get dollars?

In [ ]:
# Written by an AI assistant
def farm_revenue(deliveries):
    """Revenue per farm in dollars."""
    deliveries = deliveries.copy()
    deliveries["revenue_usd"] = deliveries["crates"] * deliveries["price_per_kg"]
    return deliveries.groupby("farm")["revenue_usd"].sum().sort_values(ascending=False)

ai_revenue = farm_revenue(deliveries)
print("The AI's revenue per farm, in dollars:")
print(ai_revenue)
print(f"The AI's total revenue: {ai_revenue.sum():,.2f} dollars")

The code ran, the numbers look like money, and the column is called `revenue_usd`. The answer is still wrong. Crates times dollars per kilogram leaves the kilograms in the unit, so the result is not dollars. The formula leaves out `kg_per_crate`. The name `revenue_usd` records what the AI meant to compute, and the formula decides what the column holds.

The next cell computes revenue from kilograms. Compare the totals, and compare which farm comes first.

In [ ]:
revenue = (deliveries["kg"] * deliveries["price_per_kg"]).groupby(deliveries["farm"]).sum().sort_values(ascending=False)
print("Revenue per farm from kilograms, in dollars:")
print(revenue)
print(f"Total revenue from kilograms: {revenue.sum():,.2f} dollars")
print("The AI's total was too small by a factor of:", round(revenue.sum() / ai_revenue.sum(), 1))
print("Top farm in the AI's answer:", ai_revenue.index[0], "| top farm by revenue from kilograms:", revenue.index[0])

## 9. Your turn: the price of one crate

Some buyers pay by the crate instead of by the kilogram. Compute the price of one crate of each crop, in dollars per crate, and store it in `price_per_crate`, with one value per crop. Before you write any code, write the units on paper: kilograms per crate times dollars per kilogram gives what?

In [ ]:
# your turn 3: one price per crop, in dollars per crate
price_per_crate = None

In [ ]:
# CHECK
assert price_per_crate is not None, "price_per_crate is still empty"
for crop, group in deliveries.groupby("crop"):
    expected = group["kg_per_crate"].iloc[0] * group["price_per_kg"].iloc[0]
    assert abs(price_per_crate[crop] - expected) < 1e-9, f"{crop}: check the units of your formula"
print("CHECK passed: dollars per crate", {c: round(float(v), 2) for c, v in price_per_crate.items()})

## 10. Your turn: check the AI's shorter way

The cell below adds up each farm's kilograms with a loop inside a loop. It is long, and every step is visible. Read it and trace farm A's first two turns in your head.

`loop_totals = {}` makes an empty dictionary, a lookup table that stores one value under each key. Here each key is a farm.

Then ask your AI assistant: *"Give me a shorter way in pandas to get the total kilograms for each farm, without a loop."* Paste its answer into the your-turn cell, store the result in `short_totals`, and run the CHECK. Two methods that agree on the same data are stronger evidence than either one alone.

In [ ]:
loop_totals = {}
for farm in ["A", "B", "C", "D"]:
    total = 0
    for kg in deliveries.loc[deliveries["farm"] == farm, "kg"]:
        total = total + int(kg)
    loop_totals[farm] = total
print(loop_totals)

In [ ]:
# your turn 4: the AI's shorter way, stored in short_totals
short_totals = None

In [ ]:
# CHECK
assert short_totals is not None, "short_totals is still empty"
for farm, total in loop_totals.items():
    assert short_totals[farm] == total, f"farm {farm}: the loop says {total}, the short way says {short_totals[farm]}"
print("CHECK passed: both methods agree for all", len(loop_totals), "farms")

## 11. Write the note for the next reader

Someone will read this program after you. In the text cell below, write three sentences for them: what the program takes in, what it produces, and where it starts. Add a fourth sentence on the mistake you found in the AI's revenue function, so the next reader does not repeat it.

*Your note here.*